In [2]:
import os

base_dir = "/content/clean_waste_data"

os.makedirs(f"{base_dir}/organic", exist_ok=True)
os.makedirs(f"{base_dir}/recyclable", exist_ok=True)

print("Created folders:")
print(f"{base_dir}/organic")
print(f"{base_dir}/recyclable")

Created folders:
/content/clean_waste_data/organic
/content/clean_waste_data/recyclable


In [4]:
!pip -q install kagglehub

In [5]:
import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "techsash/waste-classification-data"
)

print("Dataset downloaded to:", dataset_path)

for p in Path(dataset_path).rglob("*"):
    if p.is_dir() and p.name in ["O", "R"]:
        print(p)

100%|██████████| 427M/427M [00:03<00:00, 123MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1
/root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/dataset/DATASET/TRAIN/O
/root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/dataset/DATASET/TRAIN/R
/root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/dataset/DATASET/TEST/O
/root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/dataset/DATASET/TEST/R
/root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/DATASET/TRAIN/O
/root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/DATASET/TRAIN/R
/root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/DATASET/TEST/O
/root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/DATASET/TEST/R


In [7]:
from pathlib import Path

root = Path(dataset_path)

for folder in root.rglob("*"):
    if folder.is_dir() and folder.name in ["O", "R"]:
        count = sum(
            1 for p in folder.rglob("*")
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        )
        print(folder.name, ":", count, "images")

O : 12565 images
R : 9999 images
O : 1401 images
R : 1112 images
O : 12565 images
R : 9999 images
O : 1401 images
R : 1112 images


In [9]:
import shutil
import random
from pathlib import Path

random.seed(42)

# Find the original training folders
root = Path(dataset_path)
train_dir = next(
    p.parent for p in root.rglob("O")
    if p.is_dir()
    and (p.parent / "R").is_dir()
    and len(list(p.glob("*"))) > 1000
)

print("Using:", train_dir)

# Create a small dataset
small_dir = Path("/content/small_waste_data")

if small_dir.exists():
    shutil.rmtree(small_dir)

for cls in ["O", "R"]:
    source = train_dir / cls
    destination = small_dir / cls
    destination.mkdir(parents=True)

    images = [
        p for p in source.iterdir()
        if p.is_file()
        and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    chosen = random.sample(images, min(200, len(images)))

    for i, image_path in enumerate(chosen):
        shutil.copy2(image_path, destination / f"{i}{image_path.suffix.lower()}")

    print(cls, "images copied:", len(chosen))

Using: /root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/dataset/DATASET/TRAIN
O images copied: 200
R images copied: 200


In [10]:
import tensorflow as tf

IMG_SIZE = (96, 96)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    str(small_dir),
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(small_dir),
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(96, 96, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(2, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Found 400 files belonging to 2 classes.
Using 320 files for training.
Found 400 files belonging to 2 classes.
Using 80 files for validation.
Classes: ['O', 'R']
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 160ms/step - accuracy: 0.5562 - loss: 0.6759 - val_accuracy: 0.6875 - val_loss: 0.6567
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 132ms/step - accuracy: 0.5969 - loss: 0.6619 - val_accuracy: 0.7875 - val_loss: 0.5915
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 126ms/step - accuracy: 0.7156 - loss: 0.5804 - val_accuracy: 0.7875 - val_loss: 0.4995
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 130ms/step - accuracy: 0.6938 - loss: 0.5758 - val_accuracy: 0.8000 - val_loss: 0.4767
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 206ms/step - accuracy: 0.7406 - loss: 0.5350 - val_accuracy: 0.7750 - val_loss: 0.4915


In [12]:
print("Dataset path:", dataset_path)
print("Class names:", class_names)

Dataset path: /root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1
Class names: ['O', 'R']


In [14]:
print("Dataset path:", dataset_path)
print("Class names:", class_names)

Dataset path: /root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1
Class names: ['O', 'R']


In [16]:
import tensorflow as tf

IMG_SIZE = (128, 128)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    str(train_dir),
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(train_dir),
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

class_names = train_ds.class_names
print("Class names:", class_names)

Found 22564 files belonging to 2 classes.
Using 18052 files for training.
Found 22564 files belonging to 2 classes.
Using 4512 files for validation.
Class names: ['O', 'R']


In [18]:
import random
import shutil

small_dir = Path("/content/mobilenet_waste_data")

if small_dir.exists():
    shutil.rmtree(small_dir)

random.seed(42)

for cls in ["O", "R"]:
    source = train_dir / cls
    destination = small_dir / cls
    destination.mkdir(parents=True)

    images = [
        p for p in source.iterdir()
        if p.is_file()
        and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    selected = random.sample(images, min(200, len(images)))

    for i, image_path in enumerate(selected):
        shutil.copy2(image_path, destination / f"{i}.jpg")

    print(cls, "images:", len(selected))

O images: 200
R images: 200


In [20]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    str(small_dir),
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(small_dir),
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 400 files belonging to 2 classes.
Using 320 files for training.
Found 400 files belonging to 2 classes.
Using 80 files for validation.
Classes: ['O', 'R']


In [22]:
from tensorflow.keras import layers, Model

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = layers.Input(shape=(128, 128, 3))

x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.05)(x)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(2, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,260,546 (8.62 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [23]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 13s 417ms/step - accuracy: 0.6969 - loss: 0.6197 - val_accuracy: 0.7375 - val_loss: 0.4830
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 267ms/step - accuracy: 0.8188 - loss: 0.3977 - val_accuracy: 0.8250 - val_loss: 0.3676
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 265ms/step - accuracy: 0.9000 - loss: 0.2510 - val_accuracy: 0.8750 - val_loss: 0.3250
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 325ms/step - accuracy: 0.8938 - loss: 0.2529 - val_accuracy: 0.8500 - val_loss: 0.3220
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 394ms/step - accuracy: 0.9000 - loss: 0.2152 - val_accuracy: 0.9000 - val_loss: 0.2887


In [24]:
from google.colab import files
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np

uploaded = files.upload()

for filename in uploaded:
    img = load_img(filename, target_size=IMG_SIZE)
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)[0]
    index = np.argmax(prediction)

    labels = {
        "O": "Organic Waste",
        "R": "Recyclable Waste"
    }

    print("Image:", filename)
    print("Predicted class:", labels[class_names[index]])
    print(f"Confidence: {prediction[index] * 100:.2f}%")

Saving Screenshot 2026-10-24 185348.png to Screenshot 2026-10-24 185348.png
Image: Screenshot 2026-10-24 185348.png
Predicted class: Recyclable Waste
Confidence: 99.71%


In [26]:
import json

# Save the trained model
model.save("/content/waste_classifier.keras")

# Save the class names
with open("/content/class_names.json", "w") as f:
    json.dump(class_names, f)

print("Model and class names saved successfully!")

Model and class names saved successfully!


In [28]:
from google.colab import files

files.download("/content/waste_classifier.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
files.download("/content/class_names.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>